# Interactive Lden error maps — deployment cities

For each city where the Barcelona Lden model was deployed (Viladecans, Milan, Berlin, Lyon, Zaragoza) this
builds interactive HTML maps of the **Random-Forest Lden prediction error** per street segment, in two flavours:

- **Regression** — `db_diff = predicted − real Lden` (dB), `<xxx>_lden_map_db_diff_rf.html`.
- **Classification** — `class_diff = predicted − real Lden class` (0–4 bins), `<xxx>_lden_map_class_diff_rf.html`.

Blue = under-predicted, red = over-predicted, white = correct. Reads the per-city predictions from
`02_test_all_cities_lden.ipynb` and the existing street geometry (`deployment/<City>/layers/<XXX>_noise_streets.gpkg`).
Barcelona is the training city, so it is not mapped.

In [ ]:
import pandas as pd
import geopandas as gpd
import os, warnings; warnings.filterwarnings('ignore')

RES = '../results'
cities = [('Viladecans','vil','VIL'), ('Milan','mil','MIL'), ('Berlin','ber','BER'),
          ('Lyon','lyo','LYO'), ('Zaragoza','zgz','ZGZ')]

def load_roads(city, XXX):
    g = gpd.read_file(f'../../{city}/layers/{XXX}_noise_streets.gpkg')
    g['segment_id'] = g['segment_id'].astype(str)
    return g

## Regression — Lden dB error (Random Forest)

In [ ]:
for city, xxx, XXX in cities:
    preds = pd.read_csv(f'{RES}/{xxx}_lden_predictions_regre.csv')
    preds['road_id'] = preds['road_id'].astype(str)
    preds['db_diff_rf'] = (preds['pred_random'] - preds['lden_db_true']).round(2)
    merged = load_roads(city, XXX).merge(preds, left_on='segment_id', right_on='road_id', how='left')
    m = merged.to_crs(epsg=4326).explore(
        column='db_diff_rf', cmap='RdBu_r', vmin=-15, vmax=15, tiles='CartoDB positron',
        tooltip=['road_id', 'db_diff_rf', 'pred_random', 'lden_db_true', 'highway'], legend=True)
    out = f'{RES}/{xxx}_lden_map_db_diff_rf.html'; m.save(out)
    print(f'{city:11s} {merged.db_diff_rf.notna().sum():6d} segs -> {out}')

## Classification — Lden class error (Random Forest)

In [ ]:
for city, xxx, XXX in cities:
    preds = pd.read_csv(f'{RES}/{xxx}_lden_predictions_class.csv')
    preds['road_id'] = preds['road_id'].astype(str)
    preds['class_diff_rf'] = (preds['pred_random'] - preds['lden_class_true']).astype(int)
    merged = load_roads(city, XXX).merge(preds, left_on='segment_id', right_on='road_id', how='left')
    m = merged.to_crs(epsg=4326).explore(
        column='class_diff_rf', cmap='RdBu_r', vmin=-4, vmax=4, tiles='CartoDB positron',
        tooltip=['road_id', 'class_diff_rf', 'pred_random', 'lden_class_true', 'highway'], legend=True)
    out = f'{RES}/{xxx}_lden_map_class_diff_rf.html'; m.save(out)
    print(f'{city:11s} {merged.class_diff_rf.notna().sum():6d} segs -> {out}')